# Selection de variables pour le clustering flou — Benchmark comparatif

**Projet CEA-LIST** | Universite Paris-Saclay / CentraleSupelec

---

**Documents de reference du projet :**
- `article_reference_Rousselle2024` — Rousselle, Poli & Ben Abdallah (2024). *Towards an interpretable fuzzy approach to experimental design.*
- `presentation_projet.pdf` — Presentation du sujet de projet CentraleSupelec / CEA-LIST.

## 1. Introduction

### 1.1 Contexte

Le **CEA-LIST** est un institut de recherche du CEA specialise dans l'intelligence artificielle et l'apprentissage automatique, la robotique avancee, la realite virtuelle et la confiance numerique *(presentation_projet.pdf, slide 2)*.

Dans le cadre de ses recherches en **science des materiaux**, le CEA-LIST a developpe un algorithme d'aide a la decision pour l'optimisation experimentale (Rousselle, Poli & Ben Abdallah, 2024). Cet algorithme est fonde sur un **apprentissage actif utilisant le clustering flou** *(presentation_projet.pdf, slide 3)*. Son objectif est de fournir aux experimentateurs un outil **interpretable** qui :

- Selectionne la prochaine experience a mener,
- **Explique la decision** par des regles logiques,
- Ou chaque cluster correspond a une regle : Si $x_1 \wedge x_2$ alors $y = \ldots$

### 1.2 L'algorithme du CEA en 4 etapes

L'algorithme repose sur quatre etapes successives *(article_reference_Rousselle2024, Sections 2-5)* :

---

**Etape 1 — Clustering flou (Fuzzy C-Means).**

On regroupe les $n$ experiences deja realisees en $C$ clusters flous. Contrairement au clustering classique ou chaque point appartient a un seul groupe, ici chaque observation $x_i$ recoit un **degre d'appartenance** $\mu_{ci} \in [0,1]$ a chaque cluster $c$, avec la contrainte que la somme des appartenances vaut 1 :

$$\sum_{c=1}^{C} \mu_{ci} = 1 \quad \forall i$$

L'algorithme FCM (Bezdek et al., 1984) minimise la fonction objectif suivante :

$$J = \sum_{i=1}^{n} \sum_{c=1}^{C} \mu_{ci}^{m} \, \|x_i - v_c\|^2$$

ou :
- $v_c$ est le centroide du cluster $c$
- $m > 1$ est le parametre de flou (generalement $m = 2$) : plus $m$ est grand, plus les appartenances sont "douces"
- $\|x_i - v_c\|^2$ est la distance euclidienne au carre entre le point $x_i$ et le centre $v_c$

En pratique, le CEA initialise les centres de maniere **deterministe** en utilisant d'abord le clustering hierarchique (Ward), puis applique FCM pour calculer les degres d'appartenance *(article_reference_Rousselle2024, Section 3.1)*.

---

**Etape 2 — Prediction par regles de Sugeno.**

Chaque cluster $c$ genere une **formule lineaire locale** qui predit la propriete du materiau dans sa zone de l'espace :

$$\hat{y}_c(x) = \sum_{i=1}^{N} \alpha_{ci} \, x_i + \beta_c = \alpha_c^\top x + \beta_c$$

ou $\alpha_c$ sont les coefficients de regression et $\beta_c$ l'ordonnee a l'origine, specifiques au cluster $c$.

La **prediction finale** pour un point $x$ quelconque combine toutes les formules locales, ponderees par les degres d'appartenance :

$$\hat{y}(x) = \sum_{c=1}^{C} \mu_c(x) \cdot (\alpha_c^\top x + \beta_c)$$

Ainsi, dans une zone dominee par le cluster $c$ (ou $\mu_c \approx 1$), la prediction est essentiellement celle de la formule locale de $c$. Dans une zone de transition, c'est un melange des formules voisines. Les coefficients $(\alpha_c, \beta_c)$ sont optimises en minimisant l'ecart quadratique entre les valeurs predites $\hat{y}$ et les valeurs reelles $y$ des points deja echantillonnes *(article_reference_Rousselle2024, Section 3.3)*.

---

**Etape 3 — Recommandation de la prochaine experience.**

L'algorithme attribue un **score d'echantillonnage** a chaque point non encore teste :

$$S(x) = \hat{y}(x) + \lambda \cdot d(x)$$

ou :
- $\hat{y}(x)$ est la valeur predite par le modele — c'est le terme d'**exploitation** : on prefere les zones ou la propriete semble bonne
- $d(x)$ est la distance au point deja echantillonne le plus proche — c'est le terme d'**exploration** : on prefere les zones encore peu explorees
- $\lambda$ controle le compromis entre exploitation et exploration

Le point $x_{\text{next}} = \arg\max_x S(x)$ est recommande comme prochaine experience *(article_reference_Rousselle2024, Section 4)*.

---

**Etape 4 — Interpretabilite : des clusters aux regles linguistiques.**

Pour rendre le modele comprehensible, les centres des clusters sont **projetes sur chaque axe de variable**. On cree des ensembles flous triangulaires (par exemple *bas*, *moyen*, *haut*) dont les sommets correspondent a ces projections. Si deux ensembles flous sont trop proches, ils sont fusionnes.

Cela produit des **regles linguistiques** lisibles par un experimentateur humain :

> Si $x_1$ est ELEVE $\wedge$ $x_2$ est BAS $\wedge$ $\cdots$ $\wedge$ $x_N$ est MOYEN $\rightarrow$ $y$ est FORT

Le nombre de conditions dans chaque regle est egal au nombre de variables $N$ *(article_reference_Rousselle2024, Section 5)*.

C'est la que se situe le probleme.

### 1.3 Problematique

Chaque regle comporte **autant de conditions que de variables** $N$ *(presentation_projet.pdf, slide 3)*.

Avec 100 variables :
> Si $x_1 \wedge \ldots \wedge x_{100}$ alors $y = \ldots$ $\Rightarrow$ **difficilement interpretable** pour un humain

La question centrale *(presentation_projet.pdf, slide 3)* :

> **Comment selectionner les variables d'interet dans un cluster ?**

### 1.4 Objectifs du projet

Les deux objectifs, tels que definis dans la presentation initiale *(presentation_projet.pdf, slide 3)* :

1. **Determiner les methodes** de selection de variables d'interet pour le clustering flou dans la litterature.
2. **Evaluer et comparer** ces methodes sur l'approche de clustering.

### 1.5 Livrables attendus

*(presentation_projet.pdf, slide 4)* :

- **Rapport** : description des methodes de selection identifiees, description du protocole d'evaluation, discussion des resultats.
- **Code** : implementation des methodes, evaluation sur plusieurs jeux de donnees, documentation des etapes et des choix.

### 1.6 Cadre theorique

Ce benchmark s'inscrit dans le domaine de la **selection de variables pour le clustering non supervise** (*Feature Selection for Clustering*, FSC). La litterature distingue trois familles de methodes (Alelyani et al., 2018) :

- **Filter** : evaluent chaque variable independamment de l'algorithme de clustering, selon des criteres statistiques ou geometriques. Avantage : rapidite et absence de biais envers un clustering particulier. Inconvenient : ne tiennent pas compte de la qualite effective du clustering.
- **Wrapper** : utilisent un algorithme de clustering comme critere d'evaluation. Avantage : selection optimisee pour le clustering choisi. Inconvenient : cout computationnel eleve et biais envers l'algorithme utilise.
- **Embedded / Hybrid** : integrent la selection directement dans la fonction objectif du clustering. Avantage : compromis entre les deux.

La revue recente de Warkiani & Moattar (2025) complete ce cadre en abordant specifiquement la selection de variables pour les **donnees mixtes** (numeriques et categorielles), un aspect pertinent pour certains de nos datasets.

---
## 2. Presentation des bases de donnees

Dix bases de donnees ont ete fournies pour ce benchmark, reparties en deux categories : 4 issues de la science des materiaux (prefixe `MAT_`) et 6 issues de domaines reels varies (prefixe `RW_`).

### 2.1 Vue d'ensemble

| Dataset | Domaine | $n$ | $d$ | Type variables | Tache | Val. manquantes |
|---------|---------|----:|----:|---------------|-------|:---:|
| `UCI_concrete` | Materiaux | 1 030 | 8 | Continues | Regression | Non |
| `Li2023` | Materiaux | 954 | 16 | Continues + 1 discrete | Regression | Non |
| `expt_gap` | Materiaux | 4 604 | 135 | Continues + entieres | Regression | Non |
| `matbench_steels` | Materiaux | 312* | 12 | Continues | Regression | Oui* |
| `vehicle` | Silhouettes | 846 | 18 | Entieres | Classification | Non |
| `parkinsons` | Medical | 195 | 22 | Continues | Classification | Non |
| `diabetes` | Medical | 768 | 8 | Continues + discretes | Classification | Oui† |
| `breast_cancer` | Oncologie | 286 | 9 | Categorielles | Classification | Oui |
| `glass` | Criminalistique | 214 | 9 | Continues | Classification | Non |
| `covertype` | Ecologie | 581 012 | 54 | Continues + binaires | Classification | Non |

*\*115 lignes malformees (colonnes incoherentes), reduisant a 197 exploitables.*
*†Zeros physiologiquement impossibles dans Glucose, BloodPressure, SkinThickness, Insulin, BMI.*

### 2.2 Apercu des donnees

Ci-dessous, un apercu des premieres lignes de chaque dataset pour visualiser concretement la nature des donnees.

In [1]:
import pandas as pd
import os

DATA_DIR = "Datasets" + os.sep

# --- MAT_UCI_concrete ---
print("=" * 80)
print("  MAT_UCI_concrete — 8 variables continues, cible = resistance du beton (MPa)")
print("=" * 80)
df = pd.read_csv(DATA_DIR + "MAT_UCI_concrete.csv", encoding='utf-8-sig')
display(df.head())
print(f"Shape : {df.shape}\n")

  MAT_UCI_concrete — 8 variables continues, cible = resistance du beton (MPa)


,Cement (kg/m3),Blast Furnace Slag (kg/m3),Fly Ash (kg/m3),Water (kg/m3),Superplasticizer (kg/m3),Coarse Aggregate (kg/m3),Fine Aggregate (kg/m3),Age (day),Concrete compressive strength (MPa)
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


Shape : (1030, 9)



In [2]:
# --- MAT_Li2023 ---
print("=" * 80)
print("  MAT_Li2023 — 16 variables dont 'Coarse Aggerate Type' (discrete : 1-5)")
print("=" * 80)
df = pd.read_csv(DATA_DIR + "MAT_Li2023.csv", encoding='utf-8-sig')
display(df.head())
print(f"Shape : {df.shape}")
print(f"Valeurs uniques de 'Coarse Aggerate Type' : {sorted(df['Coarse Aggerate Type'].unique())}\n")

  MAT_Li2023 — 16 variables dont 'Coarse Aggerate Type' (discrete : 1-5)


,Cement Grade (MPa),Cement Density (g/cm3),Cement Content (kg/m3),Water Content (kg/m3),Fine Aggerate Content (kg/m3),Coarse Aggerate Type,Coarse Aggerate Density (g/cm3),Larger Coarse Aggregate Size (mm),Large Coarse Aggregate Size,Medium Coarse Aggregate Size,Small Coarse Aggregate Size,Larger Coarse Aggregate Size Percentage,Large Coarse Aggregate Size Percentage,Medium Coarse Aggregate Size Percentage,Small Coarse Aggregate Size Percentage,Coarse Aggerate Content (kg/m3),Concrete Strength (MPa)
0,42.5,3.10,400.0,239.0,659.0,1,2.70,2.36,4.75,8.0,16.0,5.0,77.0,12.0,6.0,999.0,40.40
1,42.5,3.10,280.0,224.0,716.0,1,2.70,2.00,4.75,4.0,8.0,38.0,48.0,6.0,8.0,1087.0,27.50
2,43.0,3.12,375.0,198.8,592.5,1,2.62,2.36,4.75,9.5,15.8,22.0,47.0,26.0,5.0,1143.8,36.84
3,43.0,3.12,400.0,200.0,572.0,1,2.62,2.36,4.75,9.5,15.8,22.0,47.0,26.0,5.0,1128.0,43.13
4,43.0,3.12,400.0,212.0,616.0,1,2.62,2.36,4.75,9.5,15.8,22.0,47.0,26.0,5.0,1196.0,38.85


Shape : (954, 17)
Valeurs uniques de 'Coarse Aggerate Type' : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]



In [3]:
# --- MAT_expt_gap ---
print("=" * 80)
print("  MAT_expt_gap — 135 variables (fractions atomiques + proprietes), cible = gap expt")
print("=" * 80)
df = pd.read_csv(DATA_DIR + "MAT_expt_gap.csv", encoding='utf-8-sig')
display(df.iloc[:5, :10])  # 10 premieres colonnes seulement
print(f"Shape : {df.shape}")
print(f"(Seules les 10 premieres colonnes sont affichees sur 136)\n")

  MAT_expt_gap — 135 variables (fractions atomiques + proprietes), cible = gap expt


,H fraction,He fraction,Li fraction,Be fraction,B fraction,C fraction,N fraction,O fraction,F fraction,Ne fraction
0,0.0,0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0
1,0.0,0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0
2,0.0,0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0
3,0.0,0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0
4,0.0,0,0.0,0.0,0.25,0.0,0.0,0.0,0.0,0


Shape : (4604, 136)
(Seules les 10 premieres colonnes sont affichees sur 136)



In [4]:
# --- MAT_matbench_steels ---
import csv as csv_mod
print("=" * 80)
print("  MAT_matbench_steels — compositions chimiques d'aciers (donnees corrompues)")
print("=" * 80)
rows = []
with open(DATA_DIR + "MAT_matbench_steels.csv", 'r', encoding='utf-8-sig') as f:
    reader = csv_mod.reader(f)
    header = next(reader)
    for row in reader:
        if len(row) == len(header):
            rows.append(row)
df = pd.DataFrame(rows, columns=header).apply(pd.to_numeric, errors='coerce')
display(df.head())
print(f"Shape apres filtrage des lignes malformees : {df.shape} (312 lignes originales, {312 - len(rows)} retirees)\n")

  MAT_matbench_steels — compositions chimiques d'aciers (donnees corrompues)


,Fe,C,Mn,Si,Cr,Ni,Mo,V,Nb,Co,Al,Ti,strength (MPa)
0,0.620,0.000953,0.000521,0.00102,0.000110,0.192000,0.017600,0.000112,0.000062,0.146000,0.003180,0.018500,2411.5
1,0.634,0.000478,0.000523,0.00102,0.000111,0.173000,0.023700,0.000113,0.000062,0.146000,0.002770,0.017600,2487.3
2,0.636,0.000474,0.000518,0.00101,0.000109,0.188000,0.008600,0.000112,0.000061,0.144000,0.002740,0.018400,2249.6
3,0.646,0.004790,0.005970,0.00492,0.135000,0.000098,0.053400,0.000113,0.000821,0.000062,0.148000,0.000640,1228.9
4,0.648,0.000453,0.000099,0.03860,0.183000,0.019500,0.000113,0.000107,0.000059,0.109000,0.000605,0.000569,1088.6


Shape apres filtrage des lignes malformees : (197, 13) (312 lignes originales, 115 retirees)



In [5]:
# --- RW_vehicle ---
print("=" * 80)
print("  RW_vehicle — 18 mesures geometriques entieres, 4 classes de vehicules")
print("=" * 80)
df = pd.read_csv(DATA_DIR + "RW_vehicle.csv", encoding='utf-8-sig')
display(df.head())
print(f"Shape : {df.shape}")
print(f"Classes : {df['class'].value_counts().to_dict()}\n")

  RW_vehicle — 18 mesures geometriques entieres, 4 classes de vehicules


,COMPACTNESS,CIRCULARITY,DISTANCE CIRCULARITY,RADIUS RATIO,PR.AXIS ASPECT RATIO,MAX.LENGTH ASPECT RATIO,SCATTER RATIO,ELONGATEDNESS,PR.AXIS RECTANGULARITY,MAX.LENGTH RECTANGULARITY,SCALED VARIANCE ALONG MAJOR AXIS,SCALED VARIANCE ALONG MINOR AXIS,SCALED RADIUS OF GYRATION,SKEWNESS ABOUT MAJOR AXIS,SKEWNESS ABOUT MINOR AXIS,KURTOSIS ABOUT MINOR AXIS,KURTOSIS ABOUT MAJOR AXIS,HOLLOWS RATIO,class
0,95.0,48,83,178,72,10,162,42,20,159,176,379,184,70,6,16,187,197,van
1,91.0,41,84,141,57,9,149,45,19,143,170,330,158,72,9,14,189,199,van
2,104.0,50,106,209,66,10,207,32,23,158,223,635,220,73,14,9,188,196,saab
3,93.0,41,82,159,63,9,144,46,19,143,160,309,127,63,6,10,199,207,van
4,85.0,44,70,205,103,52,149,45,19,144,241,325,188,127,9,11,180,183,bus


Shape : (846, 19)
Classes : {'saab': 217, 'bus': 217, 'opel': 212, 'van': 199, '204': 1}



In [6]:
# --- RW_parkinsons ---
print("=" * 80)
print("  RW_parkinsons — 22 mesures vocales, 2 classes (sain/malade)")
print("=" * 80)
df = pd.read_csv(DATA_DIR + "RW_parkinsons.csv", encoding='utf-8-sig')
display(df.head())
print(f"Shape : {df.shape}")
print(f"Classes (status) : {df['status'].value_counts().to_dict()}\n")

  RW_parkinsons — 22 mesures vocales, 2 classes (sain/malade)


,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,MDVP:Shimmer(dB),...,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE
0,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,0.426,...,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,0.626,...,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,0.482,...,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,116.676,137.871,111.366,0.00997,0.00009,0.00502,0.00698,0.01505,0.05492,0.517,...,0.08771,0.01353,20.644,1,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,116.014,141.781,110.655,0.01284,0.00011,0.00655,0.00908,0.01966,0.06425,0.584,...,0.10470,0.01767,19.649,1,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335


Shape : (195, 23)
Classes (status) : {1: 147, 0: 48}



In [7]:
# --- RW_diabetes ---
print("=" * 80)
print("  RW_diabetes — 8 variables, ATTENTION : zeros = valeurs manquantes deguisees")
print("=" * 80)
df = pd.read_csv(DATA_DIR + "RW_diabetes.csv", encoding='utf-8-sig')
display(df.head())
print(f"Shape : {df.shape}")
cols_with_fake_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for c in cols_with_fake_zeros:
    n_zeros = (df[c] == 0).sum()
    print(f"  {c:20s} : {n_zeros:3d} zeros ({100*n_zeros/len(df):.1f}%)")
print()

  RW_diabetes — 8 variables, ATTENTION : zeros = valeurs manquantes deguisees


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


Shape : (768, 9)
  Glucose              :   5 zeros (0.7%)
  BloodPressure        :  35 zeros (4.6%)
  SkinThickness        : 227 zeros (29.6%)
  Insulin              : 374 zeros (48.7%)
  BMI                  :  11 zeros (1.4%)



In [8]:
# --- RW_breast_cancer ---
print("=" * 80)
print("  RW_breast_cancer — 9 variables majoritairement categorielles")
print("=" * 80)
df = pd.read_csv(DATA_DIR + "RW_breast_cancer.csv", encoding='utf-8-sig')
display(df.head())
print(f"Shape : {df.shape}")
print(f"Types : {df.dtypes.value_counts().to_dict()}\n")

  RW_breast_cancer — 9 variables majoritairement categorielles


,class,age,menopause,tumor_size,inv_nodes,node_caps,deg_malig,breast,breast_quad,irradiat
0,no-recurrence-events,30-39,premeno,30-34,0-2,no,3,left,left_low,no
1,no-recurrence-events,40-49,premeno,20-24,0-2,no,2,right,right_up,no
2,no-recurrence-events,40-49,premeno,20-24,0-2,no,2,left,left_low,no
3,no-recurrence-events,60-69,ge40,15-19,0-2,no,2,right,left_up,no
4,no-recurrence-events,40-49,premeno,0-4,0-2,no,2,right,right_low,no


Shape : (286, 10)
Types : {dtype('O'): 9, dtype('int64'): 1}



In [9]:
# --- RW_glass ---
print("=" * 80)
print("  RW_glass — 9 mesures continues (composition chimique), 6 classes de verre")
print("=" * 80)
df = pd.read_csv(DATA_DIR + "RW_glass.csv", encoding='utf-8-sig')
display(df.head())
print(f"Shape : {df.shape}")
print(f"Classes (Type) : {df['Type'].value_counts().sort_index().to_dict()}\n")

  RW_glass — 9 mesures continues (composition chimique), 6 classes de verre


,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type
0,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


Shape : (214, 10)
Classes (Type) : {1: 70, 2: 76, 3: 17, 5: 13, 6: 9, 7: 29}



In [10]:
# --- RW_covertype (apercu seulement, trop volumineux) ---
print("=" * 80)
print("  RW_covertype — 581 012 lignes, 54 variables (10 continues + 44 binaires)")
print("=" * 80)
df = pd.read_csv(DATA_DIR + "RW_covertype.csv", encoding='utf-8-sig', nrows=5)
display(df.iloc[:, :12])  # 12 premieres colonnes
print(f"(Seules les 12 premieres colonnes et 5 lignes sont affichees)")
print(f"Nombre total de lignes : 581 012\n")

  RW_covertype — 581 012 lignes, 54 variables (10 continues + 44 binaires)


,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Wilderness_Area1,Wilderness_Area2
0,2596,51,3,258,0,510,221,232,148,6279,1,0
1,2590,56,2,212,-6,390,220,235,151,6225,1,0
2,2804,139,9,268,65,3180,234,238,135,6121,1,0
3,2785,155,18,242,118,3090,238,238,122,6211,1,0
4,2595,45,2,153,-1,391,220,234,150,6172,1,0


(Seules les 12 premieres colonnes et 5 lignes sont affichees)
Nombre total de lignes : 581 012



---
## 3. Selection et justification des 4 bases de donnees initiales

### 3.1 Criteres de selection

Le client a exprime deux axes de diversite :
- **Type de donnees** : distinguer variables continues et discretes, car les methodes de clustering fondees sur la distance euclidienne (K-Means, FCM) sont concues pour des donnees continues (Xu & Wunsch, 2005 ; Jain, 2010).
- **Domaine** : couvrir materiaux (coeur du projet CEA) et monde reel (generalisation).

En complement : un critere de **simplicite** pour cette premiere phase ($d < 20$, pas de probleme majeur de qualite, volume raisonnable).

### 3.2 Les 4 datasets retenus

| Dataset | Domaine | Type variables | Tache | $d$ | Justification |
|---------|---------|---------------|-------|----:|---------------|
| `UCI_concrete` | Materiaux | Continues | Regression | 8 | Propre, classique, sans difficulte |
| `Li2023` | Materiaux | Cont. + 1 discrete | Regression | 16 | Represente le cas mixte |
| `glass` | Monde reel | Continues | Classification | 9 | Compact, propre, multi-classes |
| `diabetes` | Monde reel | Cont. + discretes | Classification | 8 | Valeurs deguisees a traiter |

**Couverture :** 2 materiaux + 2 monde reel ; 2 purement continus + 2 avec variables discretes ; 2 regression + 2 classification.

### 3.3 Datasets ecartes et raisons

- `expt_gap` ($d=135$) : trop haute dimension pour une premiere validation ; phase 2.
- `matbench_steels` : seulement 197 observations exploitables apres filtrage.
- `breast_cancer` : variables categorielles ; les methodes spectrales sont concues pour des donnees numeriques (Alelyani et al., 2018). A traiter specifiquement en phase 2.
- `vehicle`, `parkinsons`, `covertype` : complexite moderee a elevee, phase 2.

### 3.4 Adequation type de donnees / methodes

Les methodes de clustering fondees sur la distance euclidienne (K-Means, Ward, FCM) supposent des variables continues (Xu & Wunsch, 2005 ; Jain, 2010). Pour les variables discretes, la standardisation rend les distances significatives mais l'interpretation des centroides doit etre faite avec prudence. DBSCAN est egalement sensible a la metrique mais ne presuppose pas de forme de cluster (Xu & Wunsch, 2005).

Pour les methodes de selection, les approches spectrales (Laplacian Score, SPEC, MCFS) construisent un graphe de similarite via un noyau RBF (Alelyani et al., 2018) qui suppose des distances continues. La revue de Warkiani & Moattar (2025) souligne que pour les donnees veritablement categorielles, des methodes specifiques sont necessaires.

---
## 4. Preparation des donnees

### 4.1 Pipeline

1. **Separation features / cible.** La variable cible est retiree. Elle n'est jamais vue par les algorithmes de clustering ni de selection ; elle sert uniquement a l'evaluation *a posteriori* (ARI, NMI).

2. **Nettoyage specifique :**
   - `diabetes` : remplacement des zeros physiologiquement impossibles (Glucose, BloodPressure, SkinThickness, Insulin, BMI) par `NaN`, puis imputation par la mediane.
   - `Li2023` : verification du type discret de `Coarse Aggerate Type`.

3. **Imputation** des valeurs manquantes residuelles par la mediane.

4. **Normalisation** par standardisation $z$-score :
$$z_{ij} = \frac{x_{ij} - \bar{x}_j}{s_j}$$
Indispensable pour que les distances euclidiennes soient significatives (Jain, 2010).

### 4.2 Pre-filtrage (integre a la preparation)

Deux operations reclassees de "methodes de selection" a "nettoyage des donnees", appliquees systematiquement **avant** toute methode de selection :

**Seuil de variance** — Suppression des variables quasi constantes ($\text{Var}(x_j) < \tau$, typiquement $\tau \approx 0$). Une variable constante n'apporte aucune information.

**Suppression par correlation** — Pour $|r_{jj'}| > \theta$ (typiquement $\theta = 0.95$), suppression de l'une des deux variables :
$$r_{jj'} = \frac{\sum_{i}(x_{ij}-\bar{x}_j)(x_{ij'}-\bar{x}_{j'})}{\sqrt{\sum_{i}(x_{ij}-\bar{x}_j)^2} \cdot \sqrt{\sum_{i}(x_{ij'}-\bar{x}_{j'})^2}}$$

### 4.3 Pourquoi la PCA est exclue

La PCA cree de **nouvelles variables** (combinaisons lineaires) : $z_k = \sum_j a_{kj} x_j$. Or l'objectif est de produire des regles floues portant sur les **variables originales** *(article_reference_Rousselle2024, Section 5)*. Une regle "Si $z_3$ est ELEVE" n'a aucun sens physique pour un experimentateur. De plus, comparer la PCA avec des methodes de selection n'est pas methodologiquement valide : les espaces resultants ne sont pas de meme nature.

---
## 5. Methodes de clustering

Quatre algorithmes couvrant trois paradigmes : partitionnement (K-Means), partitionnement flou (FCM), hierarchique (Ward), et densite (DBSCAN). Ce choix teste la **robustesse** des methodes de selection a travers des familles de clustering fondamentalement differentes.

### 5.1 K-Means

**Reference :** Lloyd (1982) ; MacQueen (1967). Revue detaillee dans Jain (2010).

**Principe :** minimiser l'inertie intra-cluster :
$$J_{KM} = \sum_{k=1}^{K} \sum_{x_i \in C_k} \|x_i - \mu_k\|^2$$
ou $\mu_k = \frac{1}{|C_k|}\sum_{x_i \in C_k} x_i$ est le centroide du cluster $C_k$.

**Algorithme :** (1) initialiser $K$ centroides par K-Means++, (2) affecter chaque observation au centroide le plus proche, (3) recalculer les centroides, (4) iterer jusqu'a convergence.

**Hypotheses :** clusters convexes et de taille comparable, sensible aux outliers et a l'initialisation (Jain, 2010). Convergence locale uniquement.

**Hyperparametre :** $K$. **Complexite :** $O(n \cdot K \cdot d \cdot t)$.

### 5.2 Clustering hierarchique agglomeratif (Ward)

**Reference :** Ward (1963). Classifie dans Xu & Wunsch (2005).

**Principe :** fusionner iterativement les deux clusters minimisant l'augmentation de l'inertie :
$$\Delta(C_a, C_b) = \frac{|C_a| \cdot |C_b|}{|C_a| + |C_b|} \|\mu_a - \mu_b\|^2$$

**Pertinence particuliere :** le CEA utilise Ward pour initialiser les centres du FCM *(article_reference_Rousselle2024)*. Inclure Ward evalue la selection dans ce contexte precis.

**Hyperparametre :** $K$. **Complexite :** $O(n^2 \cdot d)$.

### 5.3 Fuzzy C-Means (FCM)

**Reference :** Bezdek, Ehrlich & Full (1984).

**Principe :** generalisation floue de K-Means :
$$J_{FCM} = \sum_{i=1}^{n} \sum_{c=1}^{C} \mu_{ci}^m \|x_i - v_c\|^2 \quad \text{avec} \quad \sum_{c=1}^{C} \mu_{ci} = 1$$

Mise a jour des appartenances et des centres :
$$\mu_{ci} = \left(\sum_{c'=1}^{C} \left(\frac{\|x_i - v_c\|}{\|x_i - v_{c'}\|}\right)^{\frac{2}{m-1}}\right)^{-1} \qquad v_c = \frac{\sum_{i} \mu_{ci}^m \, x_i}{\sum_{i} \mu_{ci}^m}$$

**Pertinence :** c'est l'algorithme utilise par le CEA *(article_reference_Rousselle2024)*. Tester avec FCM evalue directement la compatibilite avec le systeme de production.

**Hyperparametres :** $C$, $m = 2$. **Complexite :** $O(n \cdot C \cdot d \cdot t)$.

### 5.4 DBSCAN

**Reference :** Ester et al. (1996). Decrit dans Xu & Wunsch (2005), Jain (2010).

**Principe :** detecter des clusters comme des regions denses separees par des regions de faible densite.

**Parametres :** $\varepsilon$ (rayon de voisinage) et `MinPts` (nombre minimum de voisins).

**Definitions :**
- *Core point* : $|N_\varepsilon(x)| \geq \texttt{MinPts}$ ou $N_\varepsilon(x) = \{y : \|x - y\| \leq \varepsilon\}$
- *Border point* : dans le $\varepsilon$-voisinage d'un *core point* mais ne satisfait pas lui-meme le critere
- *Noise point* : ni *core* ni *border* — etiquete comme outlier

**Choix de $\varepsilon$ :** methode du graphe des $k$-distances triees (*knee method*). `MinPts` fixe a $2d$.

**Gestion du bruit :** les points noise (label $= -1$) seront **exclus** du calcul des metriques internes ; leur proportion sera reportee.

**Pertinence :** DBSCAN ne genere pas de degres d'appartenance et n'est pas directement compatible avec les regles floues de Sugeno. Neanmoins, l'inclure teste la robustesse des methodes de selection a travers des paradigmes fondamentalement differents.

**Complexite :** $O(n \log n)$ avec KD-Tree.

### 5.5 Resume comparatif

| | K-Means | Ward | FCM | DBSCAN |
|---|---------|------|-----|--------|
| Paradigme | Partitionnement | Hierarchique | Part. flou | Densite |
| Forme clusters | Spherique | Spherique | Spherique | Arbitraire |
| Necessite $K$ | Oui | Oui | Oui | Non |
| Deterministe | Non (init.) | Oui | Non (init.) | Oui |
| Gere le bruit | Non | Non | Non | Oui |
| Appartenance | Dure | Dure | Floue | Dure |
| Compatible CEA | Indirect | Init. centres | **Direct** | Non |
| Complexite | $O(nKdt)$ | $O(n^2d)$ | $O(nCdt)$ | $O(n\log n)$ |

### 5.6 Choix du nombre de clusters $K$

- **Classification** : $K$ = nombre de classes connues (glass : 6, diabetes : 2).
- **Regression** (UCI_concrete, Li2023) : $K$ determine par methode du coude ou Silhouette maximale.
- **DBSCAN** : determine $K$ automatiquement via $\varepsilon$ et `MinPts`.

---
## 6. Methodes de selection de variables

Apres integration du seuil de variance et de la correlation dans la preparation, et exclusion de la PCA (Section 4.3), il reste **5 methodes** couvrant les trois familles (Alelyani et al., 2018).

### 6.1 Laplacian Score — LS (Filter)

**Reference :** He, Cai & Niyogi (2005). Decrit dans Alelyani et al. (2018).

**Principe :** evaluer la capacite de chaque variable a preserver la **structure locale** des donnees, definie par un graphe des $k$ plus proches voisins.

**Construction :**
1. Graphe $G$ : chaque point connecte a ses $k$ voisins, poids $S_{ij} = \exp(-\|x_i - x_j\|^2 / t)$
2. Matrice $D$ (diagonale, $D_{ii} = \sum_j S_{ij}$), laplacien $L = D - S$
3. Moyenne ponderee retiree : $\tilde{f}_j = f_j - \frac{f_j^\top D \mathbf{1}}{\mathbf{1}^\top D \mathbf{1}} \mathbf{1}$
4. Score :
$$\text{LS}_j = \frac{\tilde{f}_j^\top L \, \tilde{f}_j}{\tilde{f}_j^\top D \, \tilde{f}_j}$$

**Selection :** score **bas** = variable preservant la structure locale = a garder.

**Justification :** methode filter parmi les plus citees en selection non supervisee (Alelyani et al., 2018). Cas special de SPEC, ce qui permet de comparer les deux.

### 6.2 SPEC — Spectral Feature Selection (Filter)

**Reference :** Zhao & Liu (2007). Decrit dans Alelyani et al. (2018).

**Principe :** evaluer la coherence de chaque variable avec le spectre du laplacien normalise. Variante $\sigma_2$ :

$$\sigma_2(f_j) = \frac{\hat{f}_j^\top \, \mathcal{L} \, \hat{f}_j}{\hat{f}_j^\top \, \hat{f}_j}$$

ou $\mathcal{L} = D^{-1/2}(D - S)D^{-1/2}$ est le laplacien normalise.

**Justification :** generalise le Laplacian Score (Alelyani et al., 2018). Inclure les deux mesure le gain de cette generalisation.

### 6.3 Sparse K-Means (Wrapper)

**Reference :** Witten & Tibshirani (2010).

**Principe :** poids $w_j \geq 0$ par variable dans K-Means, avec regularisation $\ell_1$ :

$$\max_{w} \sum_{j=1}^{d} w_j \cdot a_j \quad \text{s.c.} \quad \|w\|_2 \leq 1, \; \|w\|_1 \leq s, \; w_j \geq 0$$

ou $a_j$ = BCSS (somme des carres inter-clusters) pour la variable $j$, et $s$ = parametre de parcimonie.

**Algorithme :** alternance K-Means pondere / *soft-thresholding* de $w$.

**Selection :** variables avec $w_j > 0.01$ retenues.

**Justification :** optimise directement un critere de clustering. Biais potentiel : liee a K-Means.

### 6.4 MCFS — Multi-Cluster Feature Selection (Filter)

**Reference :** Cai, Zhang & He (2010). Decrit dans Alelyani et al. (2018).

**Algorithme :**
1. $K$ plus petits vecteurs propres $y_1, \ldots, y_K$ du laplacien normalise
2. Pour chaque $y_l$, regression Lasso : $\min_{W_l} \|y_l - X W_l\|^2 + \alpha \|W_l\|_1$

Score : $\text{MCFS}_j = \max_l |W_{jl}|$

**Selection :** $d'$ variables avec les scores les plus eleves.

**Justification :** combine spectral + regression parcimonieuse. Surpasse souvent le Laplacian Score seul (Alelyani et al., 2018).

### 6.5 FWKM — Feature Weighting K-Means (Embedded)

**Reference :** Modha & Spangler (2003) ; Huang et al. (2005). Decrit dans Alelyani et al. (2018).

**Principe :** K-Means avec poids inversement proportionnel a la dispersion intra-cluster :

$$D_j = \sum_i \sum_l r_{il}(x_{ij} - m_{lj})^2 \qquad w_j \propto 1/D_j$$

**Compatibilite CEA :** directement adaptable au FCM en remplacant $r_{il} \in \{0, 1\}$ par $\mu_{il}^m \in [0, 1]$.

**Justification :** seule methode embedded du benchmark, completant les 3 familles. Compatible avec le pipeline du CEA.

### 6.6 Choix de $d'$ (nombre de variables retenues)

Point de depart : $d' = d/2$. Complete par une analyse de sensibilite et par le critere propre a chaque methode (poids $>$ seuil pour Sparse K-Means/FWKM, score pour les methodes filter).

### 6.7 Resume

| Methode | Famille | Principe cle | Reference |
|---------|---------|-------------|-----------|
| Laplacian Score | Filter | Structure locale (graphe) | He et al. (2005) |
| SPEC | Filter | Coherence spectrale | Zhao & Liu (2007) |
| Sparse K-Means | Wrapper | Regularisation $\ell_1$ sur K-Means | Witten & Tibshirani (2010) |
| MCFS | Filter | Spectral + Lasso | Cai et al. (2010) |
| FWKM | Embedded | Poids par variable | Huang et al. (2005) |

---
## 7. Metriques d'evaluation

### 7.1 Metriques internes (tous les datasets)

Ces metriques n'utilisent **que les clusters**, sans connaitre la cible.

#### Silhouette Score

Pour un point $x_i$ dans le cluster $C_k$ :

1. **Cohesion** : $a(i) = \frac{1}{|C_k| - 1} \sum_{\substack{x_j \in C_k \\ j \neq i}} \|x_i - x_j\|$ — distance moyenne aux autres points du meme cluster. Si $a(i)$ est petit, le point est bien au milieu de son groupe.

2. **Separation** : $b(i) = \min_{l \neq k} \frac{1}{|C_l|} \sum_{x_j \in C_l} \|x_i - x_j\|$ — distance moyenne au cluster voisin le plus proche. Si $b(i)$ est grand, le point est loin des autres groupes.

3. **Silhouette** :
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))} \in [-1, +1]$$

Silhouette globale : $\bar{s} = \frac{1}{n} \sum_i s(i)$.

Interpretation : $\bar{s} \approx +1$ = clusters tres bien separes ; $\bar{s} \approx 0$ = clusters ambigus ; $\bar{s} < 0$ = points mal assignes. **Plus c'est eleve, mieux c'est.**

#### Davies-Bouldin Index

$$DB = \frac{1}{k} \sum_{i=1}^{k} \max_{j \neq i} \frac{S_i + S_j}{\|m_i - m_j\|}$$

ou $S_i = \frac{1}{|C_i|} \sum_{x \in C_i} \|x - m_i\|$ = dispersion du cluster $i$ et $\|m_i - m_j\|$ = distance entre centres. Le ratio est grand quand les clusters sont etales et proches (mauvais). **Plus c'est bas, mieux c'est.**

#### Calinski-Harabasz Index

$$CH = \frac{\text{tr}(B_k)}{\text{tr}(W_k)} \cdot \frac{n - k}{k - 1}$$

ou $\text{tr}(B_k)$ = dispersion **inter-clusters**, $\text{tr}(W_k)$ = dispersion **intra-clusters**, $\frac{n-k}{k-1}$ = correction. **Plus c'est eleve, mieux c'est.**

### 7.2 Metriques externes (datasets de classification uniquement)

Comparent les clusters avec les vrais labels.

#### ARI — Adjusted Rand Index

Mesure la concordance entre clusters et vrais labels, ajustee pour le hasard. On examine toutes les paires de points : clusters et labels sont-ils d'accord ? $ARI = 1$ = parfait ; $ARI \approx 0$ = hasard ; $ARI < 0$ = pire que le hasard.

#### NMI — Normalized Mutual Information

$$NMI(U, V) = \frac{2 \cdot I(U; V)}{H(U) + H(V)}$$

ou $I(U;V)$ = information mutuelle, $H$ = entropie. $NMI = 1$ = parfait ; $NMI = 0$ = independance.

### 7.3 Metriques specifiques au projet

**$d'$** (nombre de variables retenues) — Metrique **centrale** du projet. Correspond a la **longueur des regles floues** de l'algorithme du CEA *(article_reference_Rousselle2024, Section 5)*. Objectif : **minimiser $d'$** en maintenant de bons scores de clustering.

**Ratio de reduction** — $d'/d$. Plus c'est bas, plus la reduction est importante.

**RMSE** (si le code CEA est disponible) — $\text{RMSE} = \sqrt{\frac{1}{n}\sum_i(\hat{y}_i - y_i)^2}$. Qualite de prediction du modele Sugeno.

---
## 8. Protocole experimental

### 8.1 Pipeline

Pour chaque dataset :
1. Preparation (Section 4) $\rightarrow$ matrice $X$ nettoyee, standardisee, pre-filtree.
2. Pour chaque methode $m \in \{\text{LS, SPEC, SK, MCFS, FWKM}\}$ : appliquer $m$ $\rightarrow$ sous-ensemble $X_m$ de $d'$ variables.
3. Pour chaque version $\in \{X_{\text{original}}, X_{\text{LS}}, X_{\text{SPEC}}, X_{\text{SK}}, X_{\text{MCFS}}, X_{\text{FWKM}}\}$ : appliquer chaque clustering $\in \{\text{K-Means, Ward, FCM, DBSCAN}\}$ et calculer les metriques.

**Nombre d'experiences :** $4 \times 6 \times 4 = 96$ (4 datasets $\times$ 6 versions $\times$ 4 clusterings).

### 8.2 Visualisations prevues

Par dataset : heatmap de correlation, variables selectionnees par methode, comparaison des metriques, visualisation 2D (t-SNE).

Vue globale : heatmap recapitulative, classement, concordance entre methodes.

### 8.3 Phases du projet

1. **Phase 1** : benchmark sur les 4 datasets retenus.
2. **Phase 2** : extension aux datasets complexes (`expt_gap`, `parkinsons`, `breast_cancer`, etc.).
3. **Phase 3** : integration avec l'algorithme complet du CEA (Sugeno, RMSE) si le code est disponible.

---
## References

### Documents de base du projet

- Rousselle, O., Poli, J.-P. & Ben Abdallah, N. (2024). *Towards an interpretable fuzzy approach to experimental design.* — `article_reference_Rousselle2024`
- Presentation du projet CentraleSupelec / CEA-LIST. — `presentation_projet.pdf`

### References academiques

- Alelyani, S., Tang, J. & Liu, H. (2018). Feature Selection for Clustering: A Review. *Data Clustering*, pp. 29-60.
- Bezdek, J.C., Ehrlich, R. & Full, W. (1984). FCM: The Fuzzy C-Means Clustering Algorithm. *Computers & Geosciences*, 10(2), pp. 191-203.
- Cai, D., Zhang, C. & He, X. (2010). Unsupervised Feature Selection for Multi-Cluster Data. *KDD*.
- Ester, M. et al. (1996). A Density-Based Algorithm for Discovering Clusters in Large Spatial Databases with Noise. *KDD*.
- He, X., Cai, D. & Niyogi, P. (2005). Laplacian Score for Feature Selection. *NIPS*.
- Huang, J.Z. et al. (2005). Automated Variable Weighting in k-Means Type Clustering. *IEEE TPAMI*, 27(5).
- Jain, A.K. (2010). Data Clustering: 50 years beyond K-means. *Pattern Recognition Letters*, 31(8), pp. 651-666.
- Min, E. et al. (2018). A Survey of Clustering With Deep Learning. *IEEE Access*, 6, pp. 35501-35524.
- Warkiani, M.E. & Moattar, M.H. (2025). A comprehensive survey on recent feature selection methods for mixed data. *Neurocomputing*, 623.
- Witten, D.M. & Tibshirani, R. (2010). A Framework for Feature Selection in Clustering. *JASA*, 105(490).
- Xu, R. & Wunsch, D. (2005). Survey of Clustering Algorithms. *IEEE Trans. Neural Networks*, 16(3), pp. 645-678.
- Zhao, Z. & Liu, H. (2007). Spectral Feature Selection for Supervised and Unsupervised Learning. *ICML*.